# XGBoost hyperparameter tuning

Notebook 05 established the XGBoost baseline and notebook 06 used the same configuration with a two-thread CPU budget. Notebook 07 already evaluated that fixed model on the 25,000-row test subset (10 false alarms and 116 missed attacks on the full test sample) and exported the model now used by batch and streaming inference. This experiment uses only the saved training and validation subsets. Model fits and predictions stay in memory.

## Confirm training inputs and baseline settings

Use the project CSV loader, binary label mapping, and ordered feature selection. Pass the selected numeric columns directly to XGBoost as in notebooks 05–07.

In [1]:
import os
import sys
from pathlib import Path

SEED = 42
THREADS = 2
if hasattr(os, "sched_getaffinity"):
    os.sched_setaffinity(0, sorted(os.sched_getaffinity(0))[:THREADS])

import numpy as np
import pandas as pd
import sklearn
import xgboost
from IPython.display import Markdown, display
from sklearn.metrics import balanced_accuracy_score
from sklearn.model_selection import ParameterGrid, StratifiedGroupKFold
from sklearn.utils.class_weight import compute_class_weight
from threadpoolctl import threadpool_limits
from xgboost import XGBClassifier

repo_root = Path.cwd().resolve()
if repo_root.name == "notebooks":
    repo_root = repo_root.parent
sys.path.insert(0, str(repo_root))

from src.batch.baseline_data import (
    ATTACK_THRESHOLD, TRAINING_PATH, VALIDATION_PATH, LABEL_MAP,
    binary_targets, load_features, select_model_features, validation_overlap,
)
from src.batch.validate import score_predictions

training, training_labels = load_features(TRAINING_PATH)
y_train = binary_targets(training_labels, "Training").to_numpy()
feature_columns = select_model_features(training)
X_train = training[feature_columns]

if (len(X_train), X_train.shape[1], ATTACK_THRESHOLD) != (100_000, 38, 0.5):
    raise ValueError("Unexpected training size, selected feature count, or threshold.")
if {"label", "replay_id", "record_id"}.intersection(feature_columns):
    raise ValueError("Model inputs include a label or identifier.")

baseline = {"max_depth": 6, "learning_rate": 0.1, "n_estimators": 100}
fixed = dict(objective="binary:logistic", tree_method="hist", device="cpu",
             random_state=SEED, n_jobs=THREADS)
classes = np.array([0, 1])

print(f"Python {sys.version.split()[0]}, scikit-learn {sklearn.__version__}, "
      f"XGBoost {xgboost.__version__}, pandas {pd.__version__}, NumPy {np.__version__}")
print(f"Baseline parameters: {baseline}")
print(f"Fixed parameters: {fixed}; attack cutoff: >= {ATTACK_THRESHOLD}")
print("Ordered feature columns:", ", ".join(feature_columns))
benign, attack = np.bincount(y_train, minlength=2)
display(Markdown(
    f"**Interpretation.** The training subset contains {benign:,} benign rows "
    f"and {attack:,} attack rows. The project maps BenignTraffic to 0 "
    f"and the {len(LABEL_MAP) - 1} documented attack labels to 1. "
    f"The 38 inputs keep the source column order and require no scaling."
))

Python 3.12.16, scikit-learn 1.9.1, XGBoost 3.4.1, pandas 3.1.1, NumPy 2.5.5
Baseline parameters: {'max_depth': 6, 'learning_rate': 0.1, 'n_estimators': 100}
Fixed parameters: {'objective': 'binary:logistic', 'tree_method': 'hist', 'device': 'cpu', 'random_state': 42, 'n_jobs': 2}; attack cutoff: >= 0.5
Ordered feature columns: flow_duration, Header_Length, Protocol Type, Duration, Rate, Drate, fin_flag_number, syn_flag_number, rst_flag_number, psh_flag_number, ack_flag_number, ack_count, syn_count, fin_count, urg_count, rst_count, HTTP, HTTPS, DNS, SSH, TCP, UDP, ARP, ICMP, IPv, Tot sum, Min, Max, AVG, Std, Tot size, IAT, Number, Magnitue, Radius, Covariance, Variance, Weight


**Interpretation.** The training subset contains 2,359 benign rows and 97,641 attack rows. The project maps BenignTraffic to 0 and the 33 documented attack labels to 1. The 38 inputs keep the source column order and require no scaling.

## Build three grouped folds and check class balance

Assign one group per exact tuple of the 38 model inputs. Shuffle with seed 42 and stratify on the binary labels while keeping each group together. Calculate balanced class weights using only the fitting rows for each fold.

In [2]:
group_ids = {}
groups = np.array([
    group_ids.setdefault(row, len(group_ids))
    for row in X_train.itertuples(index=False, name=None)
])
group_sizes = np.bincount(groups)
group_count = len(group_ids)
del group_ids

splitter = StratifiedGroupKFold(n_splits=3, shuffle=True, random_state=SEED)
folds = list(splitter.split(X_train, y_train, groups))
fold_weights = []
fold_summary = []
for number, (fit, held) in enumerate(folds, start=1):
    if np.intersect1d(groups[fit], groups[held]).size:
        raise RuntimeError("A model-input group crosses a fold boundary.")
    fit_benign, fit_attack = np.bincount(y_train[fit], minlength=2)
    held_benign, held_attack = np.bincount(y_train[held], minlength=2)
    weights = compute_class_weight("balanced", classes=classes, y=y_train[fit])
    fold_weights.append(weights)
    fold_summary.append({
        "Fold": number,
        "Fit benign": fit_benign, "Fit attack": fit_attack,
        "Held-out benign": held_benign, "Held-out attack": held_attack,
        "Benign weight": weights[0], "Attack weight": weights[1],
    })

np.testing.assert_array_equal(
    np.sort(np.concatenate([held for _, held in folds])), np.arange(len(X_train))
)
display(pd.DataFrame(fold_summary).set_index("Fold").round(6))
display(Markdown(
    f"**Interpretation.** The training rows form {group_count:,} exact input groups, "
    f"including {np.count_nonzero(group_sizes > 1):,} groups with repeated inputs. "
    "Every row is held out once, and no group crosses a fold boundary. "
    "Both classes are represented in every fold. Benign rows receive greater weight "
    "so the two classes contribute equal total fitting weight in each fold."
))

,Fit benign,Fit attack,Held-out benign,Held-out attack,Benign weight,Attack weight
Fold,,,,,,
1,1572,65094,787,32547,21.204198,0.512075
2,1573,65094,786,32547,21.191036,0.512083
3,1573,65094,786,32547,21.191036,0.512083


**Interpretation.** The training rows form 99,957 exact input groups, including 43 groups with repeated inputs. Every row is held out once, and no group crosses a fold boundary. Both classes are represented in every fold. Benign rows receive greater weight so the two classes contribute equal total fitting weight in each fold.

## Rank eight configurations using training folds

Run the fits sequentially with two XGBoost threads per fit. Score each held-out fold at 0.5 using balanced accuracy, the mean of benign recall and attack recall. Rank by mean across the three folds and report population standard deviation. If means tie exactly, prefer the original configuration, then use the displayed parameter sort order.

In [3]:
grid = ParameterGrid({
    "max_depth": [4, 6],
    "learning_rate": [0.05, 0.1],
    "n_estimators": [100, 200],
})

def predict_attack(model, inputs):
    attack_index = list(model.classes_).index(1)
    return (model.predict_proba(inputs)[:, attack_index] >= ATTACK_THRESHOLD).astype(int)

rows = []
with threadpool_limits(limits=THREADS):
    for params in grid:
        scores = []
        for (fit, held), weights in zip(folds, fold_weights, strict=True):
            model = XGBClassifier(**fixed, **params)
            model.fit(X_train.iloc[fit], y_train[fit], sample_weight=weights[y_train[fit]])
            scores.append(balanced_accuracy_score(y_train[held], predict_attack(model, X_train.iloc[held])))
        rows.append({
            **params, "Baseline": params == baseline,
            **{f"Fold {i}": score for i, score in enumerate(scores, start=1)},
            "Mean": np.mean(scores), "SD": np.std(scores, ddof=0),
        })

ranking = pd.DataFrame(rows).sort_values(
    ["Mean", "Baseline", "max_depth", "learning_rate", "n_estimators"],
    ascending=[False, False, True, True, True],
    ignore_index=True,
)
ranking.index = pd.RangeIndex(1, len(ranking) + 1, name="Rank")
score_columns = ["Fold 1", "Fold 2", "Fold 3", "Mean", "SD"]
formatted = ranking.copy()
for column in score_columns:
    formatted[column] = ranking[column].map("{:.4%}".format)
display(formatted)
best = ranking.iloc[0]
winner = {
    "max_depth": int(best["max_depth"]),
    "learning_rate": float(best["learning_rate"]),
    "n_estimators": int(best["n_estimators"]),
}
baseline_cv = ranking.loc[ranking["Baseline"], "Mean"].iloc[0]
selection = "The original configuration won." if winner == baseline else f"The winning parameters were {winner}."
display(Markdown(
    f"**Interpretation.** {selection} Its mean balanced accuracy was "
    f"{best['Mean']:.4%}, with a fold-to-fold standard deviation of {100 * best['SD']:.4f} "
    f"percentage points. The baseline averaged {baseline_cv:.4%}; "
    f"the difference was {100 * (best['Mean'] - baseline_cv):+.4f} percentage points."
))

,learning_rate,max_depth,n_estimators,Baseline,Fold 1,Fold 2,Fold 3,Mean,SD
Rank,,,,,,,,,
1,0.05,4,100,False,99.3977%,98.9408%,99.1491%,99.1625%,0.1868%
2,0.05,4,200,False,99.3177%,98.7647%,99.1731%,99.0852%,0.2342%
3,0.10,4,100,False,99.1983%,98.7616%,99.1869%,99.0489%,0.2032%
4,0.05,6,100,False,99.1363%,98.5830%,99.1869%,98.9688%,0.2735%
5,0.10,4,200,False,98.7669%,98.7671%,99.2560%,98.9300%,0.2305%
6,0.10,6,100,True,98.9513%,98.5127%,99.0790%,98.8477%,0.2426%
7,0.05,6,200,False,98.8847%,98.5111%,99.0729%,98.8229%,0.2335%
8,0.10,6,200,False,98.2350%,98.1725%,98.6814%,98.3629%,0.2266%


**Interpretation.** The winning parameters were {'max_depth': 4, 'learning_rate': 0.05, 'n_estimators': 100}. Its mean balanced accuracy was 99.1625%, with a fold-to-fold standard deviation of 0.1868 percentage points. The baseline averaged 98.8477%; the difference was +0.3149 percentage points.

## Refit and compare on validation

With the winner fixed, refit the baseline and winner on all 100,000 training rows using full-training balanced weights. Calculate both sets of predictions on the same 25,000 validation rows at 0.5. Use the project's original 46-feature exact-match mask for the second comparison. Verify that each confusion count belongs to the stated scope before calculating accuracy, precision and recall. Matrix rows are actual [benign, attack] and columns are predicted [benign, attack].

In [4]:
validation, validation_labels = load_features(VALIDATION_PATH)
y_validation = binary_targets(validation_labels, "Validation").to_numpy()
shared_patterns, matches = validation_overlap(training, validation)
if len(validation) != 25_000:
    raise ValueError("Unexpected validation size.")
X_validation = validation[feature_columns]
weights = compute_class_weight("balanced", classes=classes, y=y_train)
models = {}
with threadpool_limits(limits=THREADS):
    for name, params in {"Baseline": baseline, "CV winner": winner}.items():
        model = XGBClassifier(**fixed, **params)
        model.fit(X_train, y_train, sample_weight=weights[y_train])
        if model.get_booster().feature_names != feature_columns:
            raise ValueError("Fitted feature order differs from project feature selection.")
        models[name] = model
    predictions = {name: predict_attack(model, X_validation) for name, model in models.items()}

masks = {"All validation": np.ones(len(validation), dtype=bool), "Nonmatching only": ~matches}
results = score_predictions(y_validation, predictions, masks)

matching_rows = int(matches.sum())
scope_sizes = {scope: int(mask.sum()) for scope, mask in masks.items()}

from src.batch.validate import confusion_report, format_confusion_report

report = confusion_report(results)
extra_rates = ["Balanced accuracy", "Benign false-positive rate"]
print(f"Full-training weights: benign {weights[0]:.6f}, attack {weights[1]:.6f}")
print(f"Overlap: {matching_rows:,} validation rows across {shared_patterns:,} shared patterns.")

for scope, size in scope_sizes.items():
    comparison = report.xs(scope, level="Validation")
    if not comparison["Rows"].eq(size).all():
        raise RuntimeError("Confusion counts do not correspond to the stated validation scope.")
    display(Markdown(f"### {scope} ({size:,} rows)"))
    display(format_confusion_report(comparison.assign(Scope=scope), extra_rates))
    fp_delta = comparison.at["CV winner", "FP"] - comparison.at["Baseline", "FP"]
    fn_delta = comparison.at["CV winner", "FN"] - comparison.at["Baseline", "FN"]
    accuracy_delta = comparison.at["CV winner", "Balanced accuracy"] - comparison.at["Baseline", "Balanced accuracy"]
    display(Markdown(
        f"**Interpretation.** The winner made {abs(fp_delta):,} "
        f"{'fewer' if fp_delta < 0 else 'more'} false alarms and missed {abs(fn_delta):,} "
        f"{'fewer' if fn_delta < 0 else 'more'} attacks "
        f"relative to the baseline. Balanced accuracy changed by {100 * accuracy_delta:+.4f} percentage points."
    ))

full = results.xs("All validation", level="Validation")
baseline_errors = tuple(full.loc["Baseline", ["FP", "FN"]].astype(int))
if baseline_errors == (2, 122):
    print("Reproduced notebook 05 from predictions: 2 false alarms and 122 missed attacks.")
else:
    print(f"Notebook 05 reported (2, 122); this baseline produced {baseline_errors}.")

Full-training weights: benign 21.195422, attack 0.512080
Overlap: 25 validation rows across 25 shared patterns.


### All validation (25,000 rows)

,Model,Scope,N,TN,FP,FN,TP,Accuracy,Attack precision,Attack recall,Benign precision,Benign recall,Balanced accuracy,Benign false-positive rate
0,Baseline,All validation,25000,583,2,122,24293,99.5040%,99.9918%,99.5003%,82.6950%,99.6581%,99.5792%,0.3419%
1,CV winner,All validation,25000,584,1,212,24203,99.1480%,99.9959%,99.1317%,73.3668%,99.8291%,99.4804%,0.1709%


**Interpretation.** The winner made 1 fewer false alarms and missed 90 more attacks relative to the baseline. Balanced accuracy changed by -0.0988 percentage points.

### Nonmatching only (24,975 rows)

,Model,Scope,N,TN,FP,FN,TP,Accuracy,Attack precision,Attack recall,Benign precision,Benign recall,Balanced accuracy,Benign false-positive rate
0,Baseline,Nonmatching only,24975,581,2,122,24270,99.5035%,99.9918%,99.4998%,82.6458%,99.6569%,99.5784%,0.3431%
1,CV winner,Nonmatching only,24975,582,1,212,24180,99.1471%,99.9959%,99.1309%,73.2997%,99.8285%,99.4797%,0.1715%


**Interpretation.** The winner made 1 fewer false alarms and missed 90 more attacks relative to the baseline. Balanced accuracy changed by -0.0987 percentage points.

Reproduced notebook 05 from predictions: 2 false alarms and 122 missed attacks.


## Deployment recommendation

Use the validation counts to describe the operating trade-off at the fixed cutoff.

In [5]:
base_results = full.loc["Baseline"]
winner_results = full.loc["CV winner"]
fp_change = int(winner_results["FP"] - base_results["FP"])
fn_change = int(winner_results["FN"] - base_results["FN"])

if winner == baseline:
    recommendation = "Keep the deployed baseline. The original configuration won cross-validation."
elif fp_change <= 0 and fn_change <= 0 and (fp_change < 0 or fn_change < 0):
    recommendation = "Consider the tuned candidate for a separate deployment review."
else:
    recommendation = "Keep the deployed baseline at the current cutoff."
display(Markdown(
    f"**Recommendation.** {recommendation} On all validation rows, benign false alarms went "
    f"from {int(base_results['FP'])} to {int(winner_results['FP'])} ({fp_change:+d}), "
    f"and missed attacks went from {int(base_results['FN'])} to {int(winner_results['FN'])} ({fn_change:+d})."
))

**Recommendation.** Keep the deployed baseline at the current cutoff. On all validation rows, benign false alarms went from 2 to 1 (-1), and missed attacks went from 122 to 212 (+90).